In [1]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==================== DEFAULT CONFIGURATION ====================
DEFAULT_RECORDS_DIR = Path("records")
DEFAULT_START_ROUND = 0       # inclusive
DEFAULT_END_ROUND = None      # exclusive, None = all rounds
DEFAULT_FL_METRIC = "acc"     # acc, loss, auc, f1
# ===============================================================

In [ ]:
def load_protocol_data(
    protocol_dir: Path,
    start_round: int,
    end_round: Optional[int],
) -> Tuple[pd.DataFrame, str, int]:
    """Load CSV data and protocol metadata."""
    csv_path = protocol_dir / "compression_records.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing {csv_path}")

    df = pd.read_csv(csv_path)

    # Basic sanity checks
    for required in ("round_id", "client_id", "model_size"):
        if required not in df.columns:
            raise ValueError(f"{csv_path} missing required column: {required}")

    # Filter by round range
    df = df[df["round_id"] >= start_round]
    if end_round is not None:
        df = df[df["round_id"] < end_round]

    if len(df) == 0:
        raise ValueError(f"No data in round range [{start_round}, {end_round})")

    # Get protocol name from config (fallback to directory name)
    protocol_name = protocol_dir.name
    config_path = protocol_dir / "fl_config.json"
    if config_path.exists():
        with open(config_path, "r", encoding="utf-8") as f:
            try:
                protocol_name = json.load(f).get("codec", protocol_dir.name)
            except json.JSONDecodeError:
                # Keep fallback
                pass

    # Calculate param count (model_size in bytes, float32 = 4 bytes/param)
    # Note: assumes model_size corresponds to float32 params.
    param_count = int(df["model_size"].iloc[0] // 4)
    if param_count <= 0:
        raise ValueError("Computed non-positive param_count; check model_size column")

    return df, protocol_name, param_count


def compute_rd_metrics(df: pd.DataFrame, param_count: int) -> Dict[str, float]:
    """Average RD metrics across all rounds and clients."""
    metrics: Dict[str, float] = {}

    # Distortion metric
    if "wmape" in df.columns:
        metrics["wmape"] = float(df["wmape"].mean())
    else:
        return {}  # Skip if no distortion metric

    # Rate metric
    if "prior_rate" in df.columns:
        metrics["prior_rate"] = float(df["prior_rate"].mean())
    elif "entropy_real_rate" in df.columns:
        metrics["prior_rate"] = float(df["entropy_real_rate"].mean())
    else:
        return {}  # Skip if no rate metric

    # Meta data overhead (MB -> bits, then normalize by param count)
    if "meta_data_size" in df.columns:
        meta_mb = float(df["meta_data_size"].mean())
        metrics["meta_bits_per_param"] = (meta_mb * 8 * 1024 * 1024) / param_count
    else:
        metrics["meta_bits_per_param"] = 0.0

    return metrics


def extract_fl_progress(
    df: pd.DataFrame, metric: str
) -> Tuple[Optional[pd.DataFrame], Optional[pd.DataFrame]]:
    """Extract FL training progress for workers and global model."""
    worker_col = f"test_{metric}"
    global_col = f"global_eval_{metric}"

    # Check columns exist
    if worker_col not in df.columns or global_col not in df.columns:
        return None, None

    # Worker metrics: per-client test performance
    worker_data = df[["round_id", "client_id", worker_col]].copy()

    # Global metrics: global model evaluation (same for all clients in a round)
    global_data = df.groupby("round_id")[global_col].first().reset_index()

    return worker_data, global_data


In [ ]:
    """Plot WMAPE vs prior_rate and print meta overhead table."""
    if not all_protocols:
        print("No protocols loaded. Check your data and configuration.")
        return

    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
    colors = plt.cm.tab10(np.linspace(0, 1, len(all_protocols)))

    for idx, (protocol_name, data) in enumerate(all_protocols.items()):
        m = data["rd_metrics"]

        ax.scatter(
            m["prior_rate"],
            m["wmape"],
            s=150,
            alpha=0.8,
            color=colors[idx],
            edgecolors="black",
            linewidth=1,
            label=protocol_name,
            zorder=3,
        )

        meta_bits = m["meta_bits_per_param"]
        if meta_bits > 0.001:
            ax.annotate(
                f"{meta_bits:.3f}",
                xy=(m["prior_rate"], m["wmape"]),
                xytext=(8, -8),
                textcoords="offset points",
                fontsize=8,
                alpha=0.6,
                style="italic",
            )

    ax.set_xlabel("Prior Rate (bits/param)", fontsize=13, fontweight="bold")
    ax.set_ylabel("WMAPE (Distortion)", fontsize=13, fontweight="bold")
    ax.set_title(
        "Rate-Distortion Curve\n(averaged across rounds & clients)",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", framealpha=0.95, fontsize=10)
    ax.grid(True, alpha=0.25, linestyle="--")
    plt.tight_layout()
    plt.show()

    # Print metadata overhead table
    print(f"\n{'='*60}")
    print(f"{'Protocol':<25} | Meta Bits/Param")
    print(f"{'='*60}")
    for protocol_name, data in all_protocols.items():
        meta = data["rd_metrics"]["meta_bits_per_param"]
        print(f"{protocol_name:<25} | {meta:12.6f}")
    print(f"{'='*60}")


In [ ]:
def plot_fl_progress(all_protocols: Dict[str, Dict[str, Any]], fl_metric: str) -> None:
    """Plot worker (thin) and global (bold) training progress for a given metric."""
    if not all_protocols:
        print("No protocols loaded.")
        return

    fig, ax = plt.subplots(figsize=(13, 6), dpi=100)
    colors = plt.cm.tab10(np.linspace(0, 1, len(all_protocols)))
    plotted_any = False

    for idx, (protocol_name, data) in enumerate(all_protocols.items()):
        worker_data, global_data = extract_fl_progress(data["df"], fl_metric)

        if worker_data is None or global_data is None:
            print(f"⚠ Skipped {protocol_name}: no '{fl_metric}' data")
            continue

        plotted_any = True
        worker_col = f"test_{fl_metric}"
        global_col = f"global_eval_{fl_metric}"

        # Plot individual worker metrics (thin, transparent)
        for client_id in worker_data["client_id"].unique():
            client_df = worker_data[worker_data["client_id"] == client_id]
            ax.plot(
                client_df["round_id"],
                client_df[worker_col],
                color=colors[idx],
                alpha=0.15,
                linewidth=1,
                zorder=1,
            )

        # Plot global model metric (bold, visible)
        ax.plot(
            global_data["round_id"],
            global_data[global_col],
            color=colors[idx],
            alpha=0.9,
            linewidth=2.5,
            label=protocol_name,
            zorder=2,
        )

    if plotted_any:
        ax.set_xlabel("Round", fontsize=13, fontweight="bold")
        ax.set_ylabel(fl_metric.upper(), fontsize=13, fontweight="bold")
        ax.set_title(
            f"FL Training Progress: {fl_metric.upper()}\n(thin=workers, bold=global)",
            fontsize=14,
            fontweight="bold",
            pad=15,
        )
        ax.legend(loc="best", framealpha=0.95, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle="--")
        plt.tight_layout()
        plt.show()
    else:
        print(f"No FL progress data available for metric '{fl_metric}'")
        plt.close(fig)


In [ ]:
def load_all_protocols(records_dir: Path, start_round: int, end_round: Optional[int]) -> Dict[str, Dict[str, Any]]:
    """Scan records_dir for protocol runs and load usable ones."""
    all_protocols: Dict[str, Dict[str, Any]] = {}

    if not records_dir.exists():
        print(f"Records dir not found: {records_dir.resolve()}")
        return all_protocols

    for run_dir in sorted(records_dir.iterdir()):
        if not run_dir.is_dir():
            continue
        if not (run_dir / "compression_records.csv").exists():
            continue

        try:
            df, protocol_name, param_count = load_protocol_data(run_dir, start_round, end_round)
            rd_metrics = compute_rd_metrics(df, param_count)

            # Only include protocols with required RD metrics
            if not rd_metrics:
                print(f"⚠ Skipped {run_dir.name}: missing wmape or rate columns")
                continue

            all_protocols[protocol_name] = {
                "df": df,
                "param_count": param_count,
                "rd_metrics": rd_metrics,
            }

            num_rounds = int(df["round_id"].nunique())
            num_clients = int(df["client_id"].nunique())
            print(f"✓ {protocol_name:20s} | {num_rounds:3d} rounds × {num_clients} clients = {len(df):4d} records")

        except Exception as e:
            print(f"✗ Skipped {run_dir.name}: {e}")

    print(f"\n{'='*60}")
    print(f"Loaded {len(all_protocols)} protocol(s)")
    print(f"{'='*60}")
    return all_protocols

In [6]:
DEFAULT_RECORDS_DIR = Path("../records")
DEFAULT_START_ROUND = 10       # inclusive
DEFAULT_END_ROUND = 30      # exclusive, None = all rounds
DEFAULT_FL_METRIC = "acc"     # acc, loss, auc, f1

all_protocols = load_all_protocols(DEFAULT_RECORDS_DIR, DEFAULT_START_ROUND, DEFAULT_END_ROUND)

plot_rate_distortion(all_protocols)

plot_fl_progress(all_protocols, DEFAULT_FL_METRIC)



⚠ Skipped binary: missing wmape or rate columns
⚠ Skipped cancer_binary_w_outlier: missing wmape or rate columns
⚠ Skipped identity: missing wmape or rate columns
⚠ Skipped non_wz_learned_w_outlier: missing wmape or rate columns
⚠ Skipped trinary: missing wmape or rate columns

Loaded 0 protocol(s)
No protocols loaded. Check your data and configuration.
No protocols loaded.
